# Validação e indicadores por UF — população IBGE

**Objetivo:** validar os denominadores populacionais estaduais de 2024 e 2025, explorar diferenças entre UFs e calcular produção SIA e internações SIH por 100 mil habitantes, quando os arquivos correspondentes estiverem presentes.


## Fontes

População: [IBGE — estimativas da população](https://www.ibge.gov.br/estatisticas/sociais/populacao/9103-estimativas-de-populacao.html). As estimativas têm referência em 1º de julho. A produção ambulatorial vem do [TabNet SIA/SUS](https://tabnet.datasus.gov.br/cgi/deftohtm.exe?sia/cnv/qabr.def) e a tabulação hospitalar do [TabNet SIH/SUS](https://tabnet.datasus.gov.br/cgi/deftohtm.exe?sih/cnv/niuf.def).

Taxas abaixo são indicadores ecológicos de produção/internações por população residente, não incidência, cobertura ou risco individual.


## 1. Carregar e validar população UF

In [1]:
from pathlib import Path
import csv

BASES = [Path("../Dados/02_derivados_preliminares/IBGE"), Path("../Dados/02_derivados_preliminares/SIA"), Path("../Dados/02_derivados_preliminares/SIH"), Path("../Dados"), Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados"), Path("Dados")]
def localizar(nome):
    caminho = next((b/nome for b in BASES if (b/nome).exists()), BASES[0]/nome)
    if not caminho.exists(): raise FileNotFoundError(caminho)
    return caminho

def ler_csv(nome, encoding="utf-8-sig"):
    with localizar(nome).open(encoding=encoding, newline="") as f:
        return list(csv.DictReader(f, delimiter=";"))

pop = ler_csv("populacao_ibge_uf_2024_2025.csv")
assert len(pop) == 54, f"Esperadas 54 linhas (27 UFs x 2 anos); vieram {len(pop)}."
assert set(r["ano"] for r in pop) == {"2024","2025"}
assert len({(r["ano"],r["cod_uf"]) for r in pop}) == len(pop), "UF/ano duplicado."
for r in pop:
    r["populacao_estimada"] = int(r["populacao_estimada"])
    assert r["populacao_estimada"] > 0
print("OK: 27 UFs em cada ano, sem duplicatas e com população positiva.")


OK: 27 UFs em cada ano, sem duplicatas e com população positiva.


## 2. Conferir totais regionais e variação populacional

In [2]:
por_ano = {}
for ano in ("2024","2025"):
    recs = [r for r in pop if r["ano"] == ano]
    por_ano[ano] = sum(r["populacao_estimada"] for r in recs)
    print(f"{ano}: total das UFs = {por_ano[ano]:,}".replace(",","."))

pop24 = {r["cod_uf"]:r for r in pop if r["ano"]=="2024"}
pop25 = {r["cod_uf"]:r for r in pop if r["ano"]=="2025"}
variacoes=[]
for cod in sorted(pop24):
    a,b=pop24[cod],pop25[cod]
    assert a["sigla_uf"]==b["sigla_uf"] and a["uf"]==b["uf"]
    delta=b["populacao_estimada"]-a["populacao_estimada"]
    variacoes.append((a["uf"],delta,100*delta/a["populacao_estimada"]))
print("\nMaiores variações percentuais 2024–2025:")
for uf,delta,pct in sorted(variacoes,key=lambda x:x[2],reverse=True)[:10]:
    print(f"{uf}: {pct:.2f}% ({delta:+,})".replace(",","."))


2024: total das UFs = 212.583.750
2025: total das UFs = 213.421.037

Maiores variações percentuais 2024–2025:
Roraima: 3.07% (+21.979)
Santa Catarina: 1.60% (+128.588)
Mato Grosso: 1.49% (+57.260)
Goiás: 1.00% (+73.146)
Amazonas: 0.94% (+40.407)
Mato Grosso do Sul: 0.78% (+22.736)
Tocantins: 0.60% (+9.517)
Espírito Santo: 0.60% (+24.725)
Paraná: 0.56% (+65.852)
Pará: 0.54% (+46.890)


## 3. Taxas ecológicas de produção e internações

In [3]:
# A produção é somada por ano civil. Denominador de 1º de julho é uma aproximação
# anual; por isso, interpretar a taxa como volume administrativo relativo à população.
sia = ler_csv("hemodialise_sia_sus_uf_mes_2024_2025.csv")
sia = [r for r in sia if r.get("UF", "").strip().casefold() != "total"]
meses = [f"{ano}/{m}" for ano in (2024,2025) for m in ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]]
producao = {}
for r in sia:
    codigo_e_nome=r["UF"].strip().split(maxsplit=1)
    cod=codigo_e_nome[0]
    for ano in (2024,2025):
        producao[(cod,ano)] = sum(int(r[f"{ano}/{m}"]) for m in ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"])

print("UF | ano | produção/100 mil habitantes")
for ano in ("2024","2025"):
    resultados=[]
    for cod,rpop in (pop24 if ano=="2024" else pop25).items():
        qtd=producao[(cod,int(ano))]
        resultados.append((rpop["uf"],100000*qtd/rpop["populacao_estimada"]))
    for uf,taxa in sorted(resultados,key=lambda x:x[1],reverse=True):
        print(f"{uf} | {ano} | {taxa:.1f}")

# SIH contém metadados e hífens; análises descritivas devem usar o notebook SIH próprio.
print("\nNota: SIH agrega N17–N19 e células '-' permanecem ausentes; para taxa SIH, usar apenas valores observados e informar cobertura.")


UF | ano | produção/100 mil habitantes
Piauí | 2024 | 11747.1
Rio Grande do Norte | 2024 | 11209.7
Pernambuco | 2024 | 10824.9
Minas Gerais | 2024 | 10672.4
Ceará | 2024 | 10392.6
Alagoas | 2024 | 9771.3
Mato Grosso do Sul | 2024 | 9742.7
Bahia | 2024 | 9344.3
São Paulo | 2024 | 8731.6
Acre | 2024 | 8504.2
Espírito Santo | 2024 | 8472.4
Mato Grosso | 2024 | 8376.2
Goiás | 2024 | 8360.7
Paraíba | 2024 | 8010.3
Rio de Janeiro | 2024 | 7898.2
Paraná | 2024 | 7798.1
Sergipe | 2024 | 7753.0
Rondônia | 2024 | 7749.8
Rio Grande do Sul | 2024 | 7733.5
Maranhão | 2024 | 7646.8
Amapá | 2024 | 6980.7
Santa Catarina | 2024 | 6490.0
Roraima | 2024 | 6183.1
Pará | 2024 | 5976.4
Distrito Federal | 2024 | 5917.3
Tocantins | 2024 | 5447.1
Amazonas | 2024 | 3933.3
Piauí | 2025 | 12075.0
Rio Grande do Norte | 2025 | 11319.3
Minas Gerais | 2025 | 11074.4
Pernambuco | 2025 | 10994.9
Ceará | 2025 | 10940.5
Alagoas | 2025 | 10552.3
Bahia | 2025 | 9945.0
Mato Grosso do Sul | 2025 | 9884.7
Acre | 2025 | 9636.9

### Interpretação e limites

- Estimativas de população de 2024 e 2025 podem refletir edições metodológicas anuais; comparar mudanças com cautela.
- Produção SIA por população é um indicador agregado, não prevalência tratada nem acesso individual.
- Comparações entre UFs não controlam idade, estrutura de doença, fluxos assistenciais, oferta ou diferenças de registro.
- Para médias mensais e taxa anual, o denominador de 1º de julho é aproximação da população residente do ano.
